# 2.1.数据操作

为了能够完成各种数据操作，我们需要某种方法来存储和操作数据。通常，我们需要做两件重要的事：（1）获取数据；（2）将数据读入计算机后对其进行处理。如果没有某种方法来存储数据，那么获取数据是没有意义的。  

首先，我们介绍$n$维数组，也称为*张量*（tensor）。使用过Python中NumPy计算包的读者会对本部分很熟悉。无论使用哪个深度学习框架，它的*张量类*（在MXNet中为`ndarray`，在PyTorch和TensorFlow中为`Tensor`）都与Numpy的`ndarray`类似。但深度学习框架又比Numpy的`ndarray`多一些重要功能：首先，GPU很好地支持加速计算，而NumPy仅支持CPU计算；其次，张量类支持自动微分。这些功能使得张量类更适合深度学习。如果没有特殊说明，本书中所说的张量均指的是张量类的实例。

---

## 2.1.1.入门

本节的目标是帮助读者了解并运行一些在阅读本书的过程中会用到的基本数值计算工具。如果你很难理解一些数学概念或库函数，请不要担心。后面的章节将通过一些实际的例子来回顾这些内容。如果你已经具有相关经验，想要深入学习数学内容，可以跳过本节。

### 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"

import pypto
import torch
import torch_npu
import numpy as np

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <p style="margin: 0 0 10px 0;"><strong>首先，我们导入 torch 。请注意，虽然它被称为PyTorch，但是代码中使用 torch 而不是 pytorch 。</strong></p>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">import torch</pre>
  </div>
</details>

**张量表示一个由数值组成的数组，这个数组可能有多个维度**。具有一个轴的张量对应数学上的*向量*（vector）；具有两个轴的张量对应数学上的*矩阵*（matrix）；具有两个轴以上的张量没有特殊的数学名称。

首先，我们可以使用 `arange` 创建一个行向量 `x`。这个行向量包含以0开始的前12个整数，它们默认创建为浮点数。张量中的每个值都称为张量的 *元素*（element）。例如，张量 `x` 中有 12 个元素。除非额外指定，新的张量将存储在内存中，并采用基于CPU的计算。

In [3]:
@pypto.frontend.jit
def arange_kernel(x: pypto.Tensor([12], pypto.DT_FP32)): 
    pypto.set_vec_tile_shapes(16)
    x[:] = pypto.arange(12.0)

x_npu = torch.zeros(12, dtype=torch.float32).npu()

arange_kernel(x_npu)

print("tensor", x_npu.cpu().numpy())


tensor [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]


<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px; background-color: #ffffff;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;">PyPTO kernel 所操作的张量须通过函数参数传入，且需预先由 PyTorch 创建并放到 NPU 上，如上述代码中的<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">x_npu = torch.zeros(12, dtype=torch.float32).npu()</code>所示。</li>
      <li style="margin: 0 0 8px 0;"><code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">set_vec_tile_shapes</code>函数用于设置 vector 计算中的 TileShape 大小，即指定数据在 NPU 向量计算单元上的切分方式。TileShape 涉及 NPU 底层硬件架构的相关概念，本阶段暂不展开讨论，后续章节会作进一步说明。如需提前了解，可参阅 PyPTO 官方文档中关于 <a href="https://gitcode.com/cann/pypto/blob/0.2.0/docs/tutorials/development/tiling.md" style="color: #2563eb; text-decoration: underline; font-weight: 500;">Tiling配置</a>的内容。</li>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">x = torch.arange(12)
x</pre>
    <span style="color: #1f2937; font-family: Consolas, monospace; font-size: 13px;">tensor([0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])</span>
  </div>
</details>

**可以通过张量的`shape`属性来访问张量（沿每个轴的长度）的*形状***

In [4]:
@pypto.frontend.jit
def shape_kernel(x: pypto.Tensor([12], pypto.DT_FP32)): 
    pypto.set_vec_tile_shapes(16)
    print("tensor size:", x.shape)

shape_kernel(x_npu)

tensor size: [12]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">x.shape</pre>
    <div style="background-color: transparent; font-family: Consolas, monospace; font-size: 13px; padding: 12px; margin-top: 8px; color: #1f2937;">torch.Size([12])</div>
  </div>
</details>

PyPTO 中不支持 `numel()` 函数。如果需要获取张量的元素总数（即形状各维度大小的乘积），可以在 kernel 外部直接对 PyTorch 创建的 NPU 张量调用 `numel()` 来获取。

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <p style="margin: 12px 0;">如果只想知道张量中元素的总数，即形状的所有元素乘积，可以检查它的大小（size）。因为这里处理的是向量，所以 shape 与 size 相同。可以调用 numel() 函数来返回 input 张量的元素总数，必须在 kernel 外部直接对 PyTorch 创建的 NPU 张量调用。</p>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">x.numel()</pre>
    <div style="background-color: transparent; font-family: Consolas, monospace; font-size: 13px; padding: 12px; margin-top: 8px; color: #1f2937;">12</div>
  </div>
</details>

**要想改变一个张量的形状而不改变元素数量和元素值，可以调用`reshape`函数**。例如，可以把张量`x`从形状为（12,）的行向量转换为形状为（3,4）的矩阵。这个新的张量包含与转换前相同的值，但是它被看成一个3行4列的矩阵。要重点说明一下，虽然张量的形状发生了改变，但其元素值并没有变。注意，通过改变张量的形状，张量的大小不会改变。

In [4]:
@pypto.frontend.jit
def reshape_kernel(x: pypto.Tensor([12], pypto.DT_FP32),
              X: pypto.Tensor([3, 4], pypto.DT_FP32)): 
   pypto.set_vec_tile_shapes(16)

   X[:] = pypto.reshape(x, [3, 4])

X_npu = torch.zeros([3, 4], dtype=torch.float32).npu()

reshape_kernel(x_npu, X_npu)

print("tensor\n", X_npu.cpu().numpy(), "\n")

tensor
 [[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]] 



<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">X = x.reshape(3, 4)
X</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px;">
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])</pre>
  </div>
</details>

我们不需要通过手动指定每个维度来改变形状。也就是说，如果我们的目标形状是（高度,宽度），那么在知道宽度后，高度会被自动计算得出，不必我们自己做除法。在上面的例子中，为了获得一个3行的矩阵，我们手动指定了它有3行和4列。幸运的是，我们可以通过`-1`来调用此自动计算出维度的功能。即我们可以用`x.reshape(-1,4)`或`x.reshape(3,-1)`来取代`x.reshape(3,4)`。  

有时，我们希望**使用全0、全1、其他常量，或者从特定分布中随机采样的数字**来初始化矩阵。
我们可以创建一个形状为（2,3,4）的张量，其中所有元素都设置为0。代码如下：

In [6]:
@pypto.frontend.jit
def zero_kernel(x: pypto.Tensor([2, 3, 4], pypto.DT_FP32)): 
    pypto.set_vec_tile_shapes(2, 3, 8)
    x[:] = pypto.zeros([2, 3, 4], dtype=pypto.DT_FP32)

x1_npu = torch.empty([2, 3, 4], dtype=torch.float32).npu()

zero_kernel(x1_npu)

print("tensor\n", x1_npu.cpu().numpy())

tensor
 [[[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">torch.zeros((2, 3, 4))</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],
              [[0., 0., 0., 0.],
               [0., 0., 0., 0.],
               [0., 0., 0., 0.]]])</pre>
  </div>
</details>

同样，我们可以创建一个形状为`(2,3,4)`的张量，其中所有元素都设置为1。代码如下：

In [47]:
@pypto.frontend.jit
def one_kernel(x: pypto.Tensor([2, 3, 4], pypto.DT_FP32)): 
    pypto.set_vec_tile_shapes(2, 3, 8)
    x[:] = pypto.ones([2, 3, 4], dtype=pypto.DT_FP32)

one_kernel(x1_npu)

print("tensor\n", x1_npu.cpu().numpy())

tensor
 [[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">torch.ones((2, 3, 4))</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],
              [[1., 1., 1., 1.],
               [1., 1., 1., 1.],
               [1., 1., 1., 1.]]])</pre>
  </div>
</details>

PyPTO 中不支持 `randn()` 函数。如果需要生成服从标准正态分布的随机张量，可以先用 PyTorch 的 `torch.randn()` 创建随机数据，再将结果作为参数传入 PyPTO kernel 中使用。

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <p style="margin: 0 0 10px 0;">有时我们想通过从某个特定的概率分布中随机采样来得到张量中每个元素的值。例如，当我们构造数组来作为神经网络中的参数时，我们通常会随机初始化参数的值。以下代码创建一个形状为（3,4）的张量。其中的每个元素都从均值为0、标准差为1的标准高斯分布（正态分布）中随机采样。</p>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">torch.randn(3, 4)</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px;">tensor([[ 0.3577,  0.4575,  0.6683,  1.0181],
        [-1.1922,  0.0519, -2.7441,  0.2453],
        [ 0.1200,  0.0183,  0.6064, -0.2409]])</pre>
  </div>
</details>

PyPTO 中不支持直接将 Python 列表传入 kernel。如果需要从 Python 列表（或嵌套列表）构建张量，可以先用 PyTorch 的 `torch.tensor()` 将列表转为张量，再将结果作为参数传入 PyPTO kernel 中使用。

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <p style="margin: 0 0 10px 0;">我们还可以<strong>通过提供包含数值的Python列表（或嵌套列表），来为所需张量中的每个元素赋予确定值</strong>。在这里，最外层的列表对应于轴0，内层的列表对应于轴1。</p>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px;">tensor([[2, 1, 4, 3],
        [1, 2, 3, 4],
        [4, 3, 2, 1]])</pre>
  </div>
</details>

---

## 2.1.2.运算符

我们的兴趣不仅限于读取数据和写入数据。我们想在这些数据上执行数学运算，其中最简单且最有用的操作是*按元素*（elementwise）运算。它们将标准标量运算符应用于数组的每个元素。对于将两个数组作为输入的函数，按元素运算将二元运算符应用于两个数组中的每对位置对应的元素。我们可以基于任何从标量到标量的函数来创建按元素函数。  

在数学表示法中，我们将通过符号$f: \mathbb{R} \rightarrow \mathbb{R}$来表示*一元*标量运算符（只接收一个输入）。这意味着该函数从任何实数（$\mathbb{R}$）映射到另一个实数。同样，我们通过符号$f: \mathbb{R}, \mathbb{R} \rightarrow \mathbb{R}$表示*二元*标量运算符，这意味着该函数接收两个输入，并产生一个输出。给定同一形状的任意两个向量$\mathbf{u}$和$\mathbf{v}$和二元运算符$f$，我们可以得到向量$\mathbf{c} = F(\mathbf{u},\mathbf{v})$。具体计算方法是$c_i \gets f(u_i, v_i)$，其中$c_i$、$u_i$和$v_i$分别是向量$\mathbf{c}$、$\mathbf{u}$和$\mathbf{v}$中的元素。在这里，我们通过将标量函数升级为按元素向量运算来生成向量值$F: \mathbb{R}^d, \mathbb{R}^d \rightarrow \mathbb{R}^d$。  

对于任意具有相同形状的张量，**常见的标准算术运算符（`+`、`-`、`*`、`/`和`**`）都可以被升级为按元素运算**。我们可以在同一形状的任意两个张量上调用按元素操作。在下面的例子中，我们使用逗号来表示一个具有5个元素的元组，其中每个元素都是按元素操作的结果。

In [5]:
@pypto.frontend.jit
def elementwise_kernel(x: pypto.Tensor([4], pypto.DT_FP32),
           y: pypto.Tensor([4], pypto.DT_FP32),
           out1: pypto.Tensor([4], pypto.DT_FP32),
           out2: pypto.Tensor([4], pypto.DT_FP32),
           out3: pypto.Tensor([4], pypto.DT_FP32),
           out4: pypto.Tensor([4], pypto.DT_FP32),
           out5: pypto.Tensor([4], pypto.DT_FP32)): 
    pypto.set_vec_tile_shapes(8)

    out1[:] = pypto.add(x, y)
    out2[:] = pypto.sub(x, y)
    out3[:] = pypto.mul(x, y)
    out4[:] = pypto.div(x, y)
    out5[:] = pypto.pow(x, y)

x_npu = torch.tensor([1.0, 2.0, 4.0, 8.0], dtype=torch.float32).npu()
y_npu = torch.tensor([2.0, 2.0, 2.0, 2.0], dtype=torch.float32).npu()

out1_npu = torch.empty([4], dtype=torch.float32).npu()
out2_npu = torch.empty([4], dtype=torch.float32).npu()
out3_npu = torch.empty([4], dtype=torch.float32).npu()
out4_npu = torch.empty([4], dtype=torch.float32).npu()
out5_npu = torch.empty([4], dtype=torch.float32).npu()

elementwise_kernel(x_npu, y_npu, out1_npu, out2_npu, out3_npu, out4_npu, out5_npu)

print("x+y:", out1_npu.cpu().numpy())
print("x-y:", out2_npu.cpu().numpy())
print("x*y:", out3_npu.cpu().numpy())
print("x/y:", out4_npu.cpu().numpy())
print("x**y:", out5_npu.cpu().numpy())

x+y: [ 3.  4.  6. 10.]
x-y: [-1.  0.  2.  6.]
x*y: [ 2.  4.  8. 16.]
x/y: [0.5 1.  2.  4. ]
x**y: [ 1.  4. 16. 64.]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y  # **运算符是求幂运算</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">(tensor([ 3.,  4.,  6., 10.]),
 tensor([-1.,  0.,  2.,  6.]),
 tensor([ 2.,  4.,  8., 16.]),
 tensor([0.5000, 1.0000, 2.0000, 4.0000]),
 tensor([ 1.,  4., 16., 64.]))</pre>
  </div>
</details>

**“按元素”方式可以应用更多的计算**，包括像求幂这样的一元运算符。

In [6]:
@pypto.frontend.jit
def exp_kernel(x: pypto.Tensor([4], pypto.DT_FP32)): 
    pypto.set_vec_tile_shapes(8)
    x[:] = pypto.exp(x)

exp_kernel(x_npu)

print("tensor\n", x_npu.cpu().numpy())

tensor
 [2.7182817e+00 7.3890562e+00 5.4598148e+01 2.9809580e+03]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">torch.exp(x)</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">tensor([2.7183e+00, 7.3891e+00, 5.4598e+01, 2.9810e+03])</pre>
  </div>
</details>

除了按元素计算外，我们还可以执行线性代数运算，包括向量点积和矩阵乘法。我们将在 : [02.03_linear_algebra](02.03_linear_algebra.ipynb) 中解释线性代数的重点内容。  

**我们也可以把多个张量*连结*（concatenate）在一起**，把它们端对端地叠起来形成一个更大的张量。我们只需要提供张量列表，并给出沿哪个轴连结。下面的例子分别演示了当我们沿行（轴-0，形状的第一个元素）和按列（轴-1，形状的第二个元素）连结两个矩阵时，会发生什么情况。我们可以看到，第一个输出张量的轴-0长度（$6$）是两个输入张量轴-0长度的总和（$3 + 3$）；第二个输出张量的轴-1长度（$8$）是两个输入张量轴-1长度的总和（$4 + 4$）。

In [4]:
@pypto.frontend.jit
def concat_kernel(x: pypto.Tensor([3, 4], pypto.DT_FP32),
              y: pypto.Tensor([3, 4], pypto.DT_FP32),
              out0: pypto.Tensor([6, 4], pypto.DT_FP32),
              out1: pypto.Tensor([3, 8], pypto.DT_FP32)):

    pypto.set_vec_tile_shapes(6, 8) 
    out0[:] = pypto.concat([x, y], dim=0)
    out1[:] = pypto.concat([x, y], dim=1)

X_npu = torch.arange(12, dtype=torch.float32).reshape(3, 4).npu()
Y_npu = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]], dtype=torch.float32).npu()

out0_npu = torch.empty([6, 4], dtype=torch.float32).npu()
out1_npu = torch.empty([3, 8], dtype=torch.float32).npu()

concat_kernel(X_npu, Y_npu, out0_npu, out1_npu)

print("tensor\n", out0_npu.cpu().numpy())

print("tensor\n", out1_npu.cpu().numpy())

tensor
 [[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]
 [ 2.  1.  4.  3.]
 [ 1.  2.  3.  4.]
 [ 4.  3.  2.  1.]]
tensor
 [[ 0.  1.  2.  3.  2.  1.  4.  3.]
 [ 4.  5.  6.  7.  1.  2.  3.  4.]
 [ 8.  9. 10. 11.  4.  3.  2.  1.]]


<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <p style="margin: 0 0 10px 0;"><code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">pypto.concat</code>函数用于将输入的多个 Tensor 沿指定维度（<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">dim</code>）拼接，返回一个拼接后的 Tensor，功能与 PyTorch 中的<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">torch.cat</code>对应。需要注意的是，<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">dim</code> 只能传入单个维度。更多细节可参阅 PyPTO 官方文档中关于<a href="https://gitcode.com/cann/pypto/blob/0.2.0/docs/api/operation/pypto-concat.md" style="color: #2563eb; text-decoration: underline; font-weight: 500;">pypto.concat</a>的内容。</p>
    </ul>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 12px 0 0 20px; font-family: Consolas, monospace;">concat(tensors: List[Tensor], dim: int = 0)</pre>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">X = torch.arange(12, dtype=torch.float32).reshape((3,4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
torch.cat((X, Y), dim=0), torch.cat((X, Y), dim=1)</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))</pre>
  </div>
</details>

PyPTO kernel 内部不支持直接使用运算符对张量进行操作。在 PyTorch 中，我们可以直接写 `X == Y` 或 `X + Y`，但 PyPTO 中必须换成对应的函数调用，例如用 `pypto.eq(X, Y)` 替代 `X == Y`，用 `pypto.add(X, Y)` 替代 `X + Y`。这一点对于比较运算符（`==`、`<`、`>`）和算术运算符（`+`、`-`、`*`、`/`）同样适用。

In [8]:
@pypto.frontend.jit
def eq_kernel(X: pypto.Tensor([3, 4], pypto.DT_FP32),
             Y: pypto.Tensor([3, 4], pypto.DT_FP32),
             out: pypto.Tensor([3, 4], pypto.DT_BOOL)): 
    pypto.set_vec_tile_shapes(3, 8)
    out[:] = pypto.eq(X, Y)

out_npu = torch.empty([3, 4], dtype=torch.bool).npu()

eq_kernel(X_npu, Y_npu, out_npu)

print("X == Y:\n", out_npu.cpu().numpy())

X == Y:
 [[False  True False  True]
 [False False False False]
 [False False False False]]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <p style="margin: 12px 0;">有时，我们想<strong>通过<em> 逻辑运算符 </em>构建二元张量</strong>。</p>
    <p style="margin: 0 0 10px 0;">以<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">X == Y</code>为例：对于每个位置，如果<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">X</code>和<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">Y</code>在该位置相等，则新张量中相应项的值为1。这意味着逻辑语句<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">X == Y</code>在该位置处为真，否则该位置为0。</p>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">X == Y</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px;">tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])</pre>
  </div>
</details>

**对张量中的所有元素进行求和，会产生一个单元素张量。**

In [11]:
@pypto.frontend.jit
def sum_kernel(x: pypto.Tensor([3, 4], pypto.DT_FP32),
          mid: pypto.Tensor([3], pypto.DT_FP32),
          out: pypto.Tensor([1], pypto.DT_FP32)): 
    
    pypto.set_vec_tile_shapes(3, 8)
    mid[:] = pypto.sum(x, dim=1)
    pypto.set_vec_tile_shapes(8)
    out[:] = pypto.sum(mid, dim=0)

mid_npu = torch.empty([3], dtype=torch.float32).npu()
out_npu = torch.empty([1], dtype=torch.float32).npu()
sum_kernel(X_npu, mid_npu, out_npu)

print("tensor", out_npu.cpu().numpy())

tensor [66.]


<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;"><code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">sum()</code>函数中的<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">dim</code>参数仅支持任意单轴，如果想从多个轴进行归约，需要引入中间量，如<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">mid</code>。</li>
      <li style="margin: 0 0 8px 0;"><code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">sum()</code>函数用于对张量求和。与 PyTorch 中<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">X.sum()</code>直接对所有元素求和不同，PyPTO 的<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">sum()</code>需要显式指定沿哪个维度（<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">dim</code>）进行归约。下图直观展示了沿不同维度求和的效果（图片来自 PyPTO 官方文档 <a href="https://gitcode.com/cann/pypto/blob/master/docs/zh/api/operation/pypto-sum.md" style="color: #2563eb; text-decoration: underline; font-weight: 500;">pypto.sum</a>）。</li>
    </ul>
    <div style="padding-left:20px; margin-top:12px;">
      <div style="margin-bottom:8px;">sum按最后一个维度（<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">dim</code>=1）计算示例：</div>
      <div style="border: solid 12px #ffffff; background-color: #ffffff; width: 100%; height: 180px; display: flex; align-items: center; justify-content: center;">
        <img src="./images/sum_dim1.png" style="width: 100%; height: 100%; object-fit: contain;">
      </div>
    </div>
    <div style="padding-left:20px; margin-top:16px;">
      <div style="margin-bottom:8px;">sum按第一个维度（<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">dim</code>=0）计算示例：</div>
      <div style="border: solid 12px #ffffff; background-color: #ffffff; width: 100%; height: 180px; display: flex; align-items: center; justify-content: center;">
        <img src="./images/sum_dim0.png" style="width: 100%; height: 100%; object-fit: contain;">
      </div>
    </div>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">X.sum()</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">tensor(66.)</pre>
  </div>
</details>

---

## 2.1.3.广播机制

在上面的部分中，我们看到了如何在相同形状的两个张量上执行按元素操作。在某些情况下，**即使形状不同，我们仍然可以通过调用*广播机制*（broadcasting mechanism）来执行按元素操作**。这种机制的工作方式如下：

1. 通过适当复制元素来扩展一个或两个数组，以便在转换之后，两个张量具有相同的形状；
2. 对生成的数组执行按元素操作。

在大多数情况下，我们将沿着数组中长度为1的轴进行广播。

由于`a`和`b`分别是$3\times1$和$1\times2$矩阵，如果让它们相加，它们的形状不匹配。我们将两个矩阵*广播*为一个更大的$3\times2$矩阵，如下所示：矩阵`a`将复制列，矩阵`b`将复制行，然后再按元素相加。


In [19]:
@pypto.frontend.jit
def broad_add_kernel(a: pypto.Tensor([3, 1], pypto.DT_FP32),
          b: pypto.Tensor([1, 2], pypto.DT_FP32),
          out: pypto.Tensor([3, 2], pypto.DT_FP32)): 
        pypto.set_vec_tile_shapes(3, 8)
        out[:] = pypto.add(a, b)
        
a_npu = torch.arange(3, dtype=torch.float32).reshape(3, 1).npu()
b_npu = torch.arange(2, dtype=torch.float32).reshape(1, 2).npu()
out_npu = torch.empty([3, 2], dtype=torch.float32).npu()

broad_add_kernel(a_npu, b_npu, out_npu)

print("tensor\n", a_npu.cpu().numpy())
print("tensor\n", b_npu.cpu().numpy())
print("tensor\n", out_npu.cpu().numpy())

tensor
 [[0.]
 [1.]
 [2.]]
tensor
 [[0. 1.]]
tensor
 [[0. 1.]
 [1. 2.]
 [2. 3.]]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">a = torch.arange(3).reshape((3, 1))
b = torch.arange(2).reshape((1, 2))
a, b, a + b</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">(tensor([[0],
         [1],
         [2]]),
 tensor([[0, 1]]))
 tensor([[0, 1],
        [1, 2],
        [2, 3]])</pre>
  </div>
</details>

---

## 2.1.4.索引和切片

就像在任何其他Python数组中一样，张量中的元素可以通过索引访问。与任何Python数组一样：第一个元素的索引是0，最后一个元素索引是-1；可以指定范围以包含第一个元素和最后一个之前的元素。  

如下所示，我们**可以用`[-1]`选择最后一个元素，可以用`[1:3]`选择第二个和第三个元素**：

In [7]:
@pypto.frontend.jit
def index_kernel(x: pypto.Tensor([3, 4], pypto.DT_FP32),
                      out1: pypto.Tensor([4], pypto.DT_FP32),
                      out2: pypto.Tensor([2, 4], pypto.DT_FP32)): 

        pypto.set_vec_tile_shapes(3, 8)
        out1[:] = x[-1]
        out2[:] = x[1:3]

out1_npu = torch.zeros([4], dtype=torch.float32).npu()
out2_npu = torch.zeros([2, 4], dtype=torch.float32).npu()

index_kernel(X_npu, out1_npu, out2_npu)

print("tensor\n", out1_npu.cpu().numpy())
print("tensor\n", out2_npu.cpu().numpy())

tensor
 [ 8.  9. 10. 11.]
tensor
 [[ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">X[-1], X[1:3]</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">(tensor([ 8.,  9., 10., 11.]),
 tensor([[ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))</pre>
  </div>
</details>

**除读取外，我们还可以通过指定索引来将元素写入矩阵。**

In [5]:
@pypto.frontend.jit
def index_write_kernel(x: pypto.Tensor([3, 4], pypto.DT_FP32),
                  value: pypto.Tensor([1, 1], pypto.DT_FP32)):
        pypto.set_vec_tile_shapes(3, 8) 
        x[1: 2, 2: 3] = value
        
value_npu = torch.tensor([[9.0]], dtype=torch.float32).npu()

index_write_kernel(X_npu, value_npu)

print("tensor\n", X_npu.cpu().numpy())

tensor
 [[ 0.  1.  2.  3.]
 [ 4.  5.  9.  7.]
 [ 8.  9. 10. 11.]]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">X[1, 2] = 9
X</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  9.,  7.],
        [ 8.,  9., 10., 11.]])</pre>
  </div>
</details>

如果我们想**为多个元素赋值相同的值，我们只需要索引所有元素，然后为它们赋值**。例如，`[0:2, :]`访问第1行和第2行，其中“:”代表沿轴1（列）的所有元素。虽然我们讨论的是矩阵的索引，但这也适用于向量和超过2个维度的张量。

In [6]:
@pypto.frontend.jit
def index_write_more_kernel(x: pypto.Tensor([3, 4], pypto.DT_FP32),                      
                       value: pypto.Tensor([2, 4], pypto.DT_FP32)): 
        pypto.set_vec_tile_shapes(3, 8)
        x[0:2, :] = value 

value_npu = torch.full((2, 4), 12.0, dtype=torch.float32).npu()

index_write_more_kernel(X_npu, value_npu)
print("tensor\n", X_npu.cpu().numpy())

tensor
 [[12. 12. 12. 12.]
 [12. 12. 12. 12.]
 [ 8.  9. 10. 11.]]


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">X[0:2, :] = 12
X</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">tensor([[12., 12., 12., 12.],
        [12., 12., 12., 12.],
        [ 8.,  9., 10., 11.]])</pre>
  </div>
</details>

---

## 2.1.5.节省内存

**运行一些操作可能会导致为新结果分配内存**。例如，如果我们用`Y = X + Y`，我们将取消引用`Y`指向的张量，而是指向新分配的内存处的张量。  

在下面的例子中，我们用Python的`id()`函数演示了这一点，它给我们提供了内存中引用对象的确切地址。运行`Y = Y + X`后，我们会发现`id(Y)`指向另一个位置。这是因为Python首先计算`Y + X`，为结果分配新的内存，然后使`Y`指向内存中的这个新位置。

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">before = id(Y)
Y = Y + X
id(Y) == before</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">False</pre>
  </div>
</details>

这可能是不可取的，原因有两个：

1. 首先，我们不想总是不必要地分配内存。在机器学习中，我们可能有数百兆的参数，并且在一秒内多次更新所有参数。通常情况下，我们希望原地执行这些更新；
2. 如果我们不原地更新，其他引用仍然会指向旧的内存位置，这样我们的某些代码可能会无意中引用旧的参数。

幸运的是，**执行原地操作**非常简单。我们可以使用切片表示法将操作的结果分配给先前分配的数组，例如`Y[:] = <expression>`。为了说明这一点，我们首先创建一个新的矩阵`Z`，其形状与另一个`Y`相同，使用`zeros_like`来分配一个全$0$的块。

In [22]:
@pypto.frontend.jit
def inplace_kernel(x: pypto.Tensor([3, 4], pypto.DT_FP32),
          y: pypto.Tensor([3, 4], pypto.DT_FP32),
          Z: pypto.Tensor([3, 4], pypto.DT_FP32)):

    pypto.set_vec_tile_shapes(3, 8)
    Z[:] = pypto.add(x, y)

Z = torch.zeros_like(Y_npu)
before = id(Z)
print("id(Z):" ,before)

inplace_kernel(X_npu, Y_npu, Z)

print("id(Z):",id(Z))

id(Z): 281467565272976
id(Z): 281467565272976


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">Z = torch.zeros_like(Y)
print('id(Z):', id(Z))
Z[:] = X + Y
print('id(Z):', id(Z))</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">id(Z): 281468419962928
id(Z): 281468419962928</pre>
  </div>
</details>

**如果在后续计算中没有重复使用`X`，我们也可以使用`X[:] = X + Y`或`X += Y`来减少操作的内存开销。**

In [15]:
@pypto.frontend.jit
def inplace_kernel(x: pypto.Tensor([3, 4], pypto.DT_FP32),
              y: pypto.Tensor([3, 4], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(3, 8)
    x[:] = pypto.add(x, y)

before = id(X_npu)

inplace_kernel(X_npu, Y_npu)
is_inplace = (id(X_npu) == before)

print(is_inplace)

True


<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击展开 / 折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;">这段代码对应了<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">X[:] = X + Y</code>的思路，通过<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">Y[:] = pypto.add(X, Y)</code>将计算结果直接写回预先分配的张量<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">Y</code>，实现了原地操作</li>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击展开 / 折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">before = id(X)
X += Y
id(X) == before</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px;">True</pre>
  </div>
</details>

---

## 2.1.6.转换为其他Python对象


与 PyTorch 不同，PyPTO 张量存储于 NPU，而 NumPy 数组驻留在 CPU 内存中，二者处于不同的地址空间，无法共享底层存储。因此 PyPTO 不支持类似 `torch.from_numpy()` 的零拷贝互转接口，如果需要在两者之间转换数据，须借助 PyTorch 张量作为中转。


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <p style="margin: 0 0 10px 0;">将深度学习框架定义的张量<strong>转换为NumPy张量（<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">ndarray</code>）</strong>很容易，反之也同样容易。torch张量和numpy数组将共享它们的底层内存，就地操作更改一个张量也会同时更改另一个张量。</p>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">A = X.numpy()
B = torch.tensor(A)
type(A), type(B)</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">(numpy.ndarray, torch.Tensor)</pre>
  </div>
</details>

类似地，`item()`、`float()`、`int()` 等将张量转为 Python 标量的方法均为 CPU 端操作，无法直接作用于 NPU 上的 PyPTO 张量。如果需要获取 Python 标量值，须先将张量通过 `.cpu()` 拷贝至 CPU，再调用上述方法，例如：`x_npu.cpu().item()`。

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <p style="margin: 0 0 10px 0;">要<strong>将大小为1的张量转换为Python标量</strong>，我们可以调用<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">item</code>函数或Python的内置函数。</p>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">a = torch.tensor([3.5])
a, a.item(), float(a), int(a)</pre>
    <pre style="background-color: transparent; color: #1f2937; border-radius: 0; padding: 12px; font-family: Consolas, monospace; font-size: 13px; margin-top: 8px; white-space:pre-wrap;">(tensor([3.5000]), 3.5, 3.5, 3)</pre>
  </div>
</details>

---

## 2.1.7.小结

* 深度学习存储和操作数据的主要接口是张量（$n$维数组）。它提供了各种功能，包括基本数学运算、广播、索引、切片、内存节省和转换其他Python对象。
---
## 2.1.8.练习

1. 运行本节中的代码。将本节中的条件语句`X == Y`更改为`X < Y`或`X > Y`，然后看看你可以得到什么样的张量。
1. 用其他形状（例如三维张量）替换广播机制中按元素操作的两个张量。结果是否与预期相同？

参考答案见 [answers/02.01_reference_answer](answers/02.01_reference_answer.ipynb)，其中同时给出了 PyTorch 与 PyPTO 两种实现方式，供对照学习。